## Libraries

In [1]:
import geopandas as gpd
import requests
import os
import pandas as pd
import time

# List of all municipalities and geographic coordinates

From OverPass Turbo (https://overpass-turbo.eu/) with the following query:

[out:json][timeout:60];

// area provincia di Sondrio
{{geocodeArea:Provincia di Sondrio}}->.searchArea;

// comuni (relazioni admin_level=8)
relation["boundary"="administrative"]["admin_level"="8"](area.searchArea)->.comuni;

// capoluoghi (admin_centre dei comuni)
node(r.comuni:"admin_centre");

// output con coordinate
out body;

In [2]:
os.getcwd()
os.chdir("raw_data")

Municipalities include:
* name (name of the town, not of the municipality. *E.g. municipality of Valdisotto = Cepina in this table*)
* local name
* category (town, village...)
* elevation (m)
* geometry (coordinates)

In [55]:
municipalities = gpd.read_file("municipalities.geojson")
municipalities = municipalities[["name", "loc_name", "place", "ele", "geometry"]]
municipalities["lat"] = municipalities["geometry"].y
municipalities["lon"] = municipalities["geometry"].x
municipalities.drop(columns="geometry", inplace=True)
municipalities.head()

,name,loc_name,place,ele,lat,lon
0,Sondrio,Sundri,town,307,46.171208,9.871622
1,Rasura,Rèsüra,village,800,46.100665,9.552971
2,Ponte in Valtellina,Pùnt,village,486,46.176767,9.981882
3,Poggiridenti,Pendulasch,village,564,46.175662,9.926246
4,Novate Mezzola,Nuàa,village,212,46.220465,9.447221


Data structure:
* elevation = incorrect data type (string instead of integer)
* 77 municipalities
    * 14 missing "local names"
    * 4 missing elevations
    * no duplicates 

In [56]:
print(municipalities.info())
print("################# Null values ###################")
print(municipalities.isnull().sum())

print("################# Duplicated values ###################")
print(municipalities.duplicated().sum())
print("################# Check duplicated ###################")
print(municipalities[municipalities.duplicated()])

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 77 entries, 0 to 76
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   name      77 non-null     str    
 1   loc_name  63 non-null     str    
 2   place     77 non-null     str    
 3   ele       73 non-null     str    
 4   lat       77 non-null     float64
 5   lon       77 non-null     float64
dtypes: float64(2), str(4)
memory usage: 3.7 KB
None
################# Null values ###################
name         0
loc_name    14
place        0
ele          4
lat          0
lon          0
dtype: int64
################# Duplicated values ###################
0
################# Check duplicated ###################
Empty DataFrame
Columns: [name, loc_name, place, ele, lat, lon]
Index: []


In [57]:
# municipalities[municipalities["ele"].isnull()]
municipalities.loc[municipalities["name"] == "Villa di Tirano", "ele"] = "406"
municipalities.loc[municipalities["name"] == "Sernio", "ele"] = "632"
municipalities.loc[municipalities["name"] == "Campodolcino", "ele"] = "1071"
municipalities.loc[municipalities["name"] == "Regoledo", "ele"] = "231"
municipalities["ele"] = municipalities["ele"].astype(int)

In [62]:
municipalities.to_csv("../processed_data/municipalities.csv", index=False)

# Weather data

From the website (https://open-meteo.com/en/docs/historical-weather-api)

In [3]:
municipalities_api = pd.read_csv("../processed_data/municipalities.csv")
municipalities_api = municipalities_api[["name", "lat", "lon"]]

I had to deal with limits of the free API:
* 600 calls/min
* 5k calls/hour
* 10k calls/day
* 300k calls/month 

According to the calculator (https://open-meteo.com/en/pricing):
* 11 variables (8 hourly and 3 daily)
* 1 year
* 1 model
* 77 locations

costs 2208.3 API calls, so I can ask for 2 years per hour and 4 years per day

In [4]:
base_url = "https://archive-api.open-meteo.com/v1/archive"

def download_data(lat, lon):
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": "2024-01-01",
        "end_date": "2025-12-31",
        "hourly": (
            "temperature_2m,apparent_temperature,rain,"
            "snowfall,wind_speed_10m,wind_direction_10m,"
            "wind_gusts_10m,weather_code,snow_depth"
        ),
        "daily": (
            "sunrise,sunset,daylight_duration"
        ),
        "timezone": "Europe/Rome"
    }

    while True:
        r = requests.get(base_url, params=params, timeout=30)

        if r.status_code == 429:
            print("Error 429: Minutely API request limit exceeded. Waiting 61 seconds")
            time.sleep(61)
            print("Continuing download")
            continue

        if not r.ok:
            print("Status code:", r.status_code)
            print("API error:", r.text)
            print("Request URL:", r.url)

        r.raise_for_status()
        return r.json()

hourly_results = []
daily_results = []

for i, row in municipalities_api.iterrows():
    try:
        print(f"Downloading data: {row['name']} ({i+1}/{len(municipalities_api)})")

        data = download_data(row["lat"], row["lon"])

        hourly_df = pd.DataFrame(data["hourly"])
        hourly_df["name"] = row["name"]
        hourly_results.append(hourly_df)

        daily_df = pd.DataFrame(data["daily"])
        daily_df["name"] = row["name"]
        daily_results.append(daily_df)

        time.sleep(1.2 + i * 0.05)

    except Exception as e:
        print(f"Error on {row['name']}: {e}")

        time.sleep(3)

hourly_final = pd.concat(hourly_results, ignore_index=True)
hourly_final.to_csv(r"hourly_weather_data_2024_25.csv", index=False)

daily_final = pd.concat(daily_results, ignore_index=True)
daily_final.to_csv(r"daily_weather_data_2024_25.csv", index=False)

print("Download completed")

Error 429: Minutely API request limit exceeded. Waiting 61 seconds
Continuing download
Error 429: Minutely API request limit exceeded. Waiting 61 seconds
Continuing download
Error 429: Minutely API request limit exceeded. Waiting 61 seconds
Continuing download
Error 429: Minutely API request limit exceeded. Waiting 61 seconds
Continuing download
Error 429: Minutely API request limit exceeded. Waiting 61 seconds
Continuing download
Error 429: Minutely API request limit exceeded. Waiting 61 seconds
Continuing download
Download completed
